# 3DGS to Mesh Extraction & Evaluation

This notebook demonstrates how to convert a trained **3D Gaussian Splatting (3DGS)** model (`3DGS_Result.ply`) into a clean **3D Polygonal Triangle Mesh** (`.ply` / `.obj`).

### Workflow Pipeline
1. **Parse & Filter 3DGS Attributes**: Extract 3D Gaussian centers $(x, y, z)$, evaluate opacities $\sigma$, spherical harmonics $f_{dc}$ to RGB colors, and calculate splat scales.
2. **Point Cloud Preprocessing**: Remove floating artifacts, filter low-opacity/outlier splats, and estimate consistent surface normals.
3. **Mesh Reconstruction Algorithms**:
   - **Screened Poisson Surface Reconstruction**: Water-tight, smooth surface estimation with low-density trimming.
   - **Ball-Pivoting Algorithm (BPA)**: Delaunay-based rolling ball reconstruction preserving sharp geometric features.
4. **Post-Processing**: Non-manifold edge cleanup, mesh smoothing, and vertex color transfer.
5. **Interactive 3D Visualization & Export**: Interactive Plotly/Open3D inspection and export to standard formats (`.ply`, `.obj`).

In [1]:
# Step 0: Imports and Environment Setup
import os
import numpy as np
import open3d as o3d
from plyfile import PlyData
import matplotlib.pyplot as plt
import plotly.graph_objects as go

print(f"Open3D version: {o3d.__version__}")
print("Libraries loaded successfully!")

Open3D version: 0.19.0
Libraries loaded successfully!


In [2]:
# Step 1: Define File Paths
INPUT_PLY_PATH = "../outputs/export_splat_clean 3/3DGS_Result.ply"
OUTPUT_DIR = "../outputs/mesh_from_3dgs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.exists(INPUT_PLY_PATH), f"File not found: {INPUT_PLY_PATH}"
print(f"Found input 3DGS file: {INPUT_PLY_PATH} ({os.path.getsize(INPUT_PLY_PATH) / 1e6:.2f} MB)")

Found input 3DGS file: ../outputs/export_splat_clean 3/3DGS_Result.ply (47.61 MB)


In [3]:
# Step 2: Parse 3DGS PLY and Convert Attributes
print("Reading 3DGS PLY...")
plydata = PlyData.read(INPUT_PLY_PATH)
vertex = plydata["vertex"]
total_splats = len(vertex.data)
print(f"Total 3D Gaussians loaded: {total_splats:,}")

# 1. Splat positions (x, y, z)
xyz = np.stack([vertex["x"], vertex["y"], vertex["z"]], axis=-1).astype(np.float64)

# 2. Opacity: 3DGS stores opacity in logit space; convert with sigmoid: 1 / (1 + exp(-x))
raw_opacity = np.asarray(vertex["opacity"]).astype(np.float64)
opacity = 1.0 / (1.0 + np.exp(-raw_opacity))

# 3. Colors: Convert 0-order spherical harmonics (f_dc) to RGB [0, 1]
SH_C0 = 0.28209479177387814
rgb = np.stack([vertex["f_dc_0"], vertex["f_dc_1"], vertex["f_dc_2"]], axis=-1).astype(np.float64) * SH_C0 + 0.5
rgb = np.clip(rgb, 0.0, 1.0)

# 4. Scales: 3DGS scales are stored in log-space
scale = np.exp(np.stack([vertex["scale_0"], vertex["scale_1"], vertex["scale_2"]], axis=-1).astype(np.float64))
mean_scale = np.mean(scale, axis=-1)

print(f"Bounding box min: {xyz.min(axis=0)}")
print(f"Bounding box max: {xyz.max(axis=0)}")
print(f"Opacity percentiles (25%, 50%, 75%): {np.percentile(opacity, [25, 50, 75]).round(3)}")

Reading 3DGS PLY...
Total 3D Gaussians loaded: 191,983
Bounding box min: [-0.07507717 -0.01113247 -0.1727616 ]
Bounding box max: [0.19335219 0.23214076 0.11137202]
Opacity percentiles (25%, 50%, 75%): [0.505 0.861 0.978]


In [4]:
# Step 3: Filter Points and Prepare Clean Point Cloud
# Gaussians with low opacity are often semi-transparent fog or background noise
OPACITY_THRESHOLD = 0.5
# Gaussians with abnormally large scales are often floaters / sky artifacts
SCALE_MAX_THRESHOLD = np.percentile(mean_scale, 98)

valid_mask = (opacity >= OPACITY_THRESHOLD) & (mean_scale <= SCALE_MAX_THRESHOLD)
print(f"Retained {np.sum(valid_mask):,} / {total_splats:,} points ({np.mean(valid_mask)*100:.1f}%)")

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz[valid_mask])
pcd.colors = o3d.utility.Vector3dVector(rgb[valid_mask])

# Statistical Outlier Removal to eliminate remaining isolated floaters
cl, ind = pcd.remove_statistical_outlier(nb_neighbors=25, std_ratio=1.5)
pcd_clean = pcd.select_by_index(ind)
print(f"Points after outlier removal: {len(pcd_clean.points):,}")

# Estimate surface normals
print("Estimating surface normals...")
pcd_clean.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.015, max_nn=30)
)
pcd_clean.orient_normals_consistent_tangent_plane(k=15)

Retained 141,114 / 191,983 points (73.5%)
Points after outlier removal: 131,582
Estimating surface normals...


In [5]:
# Step 4: Reconstruction Method A - Screened Poisson Surface Reconstruction
print("Running Screened Poisson Surface Reconstruction...")
# depth=8 or 9 balances detail and speed (depth=9 is finer, depth=8 is fast)
poisson_mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd_clean, depth=8, linear_fit=True
)
densities = np.asarray(densities)

# Poisson creates a bounding hull; trim low-density vertices
density_threshold = np.percentile(densities, 5)
vertices_to_remove = densities < density_threshold
poisson_mesh.remove_vertices_by_mask(vertices_to_remove)

# Clean up non-manifold elements
poisson_mesh.remove_degenerate_triangles()
poisson_mesh.remove_duplicated_triangles()
poisson_mesh.remove_duplicated_vertices()
poisson_mesh.remove_non_manifold_edges()

print(f"Poisson Mesh - Vertices: {len(poisson_mesh.vertices):,}, Triangles: {len(poisson_mesh.triangles):,}")

Running Screened Poisson Surface Reconstruction...
Poisson Mesh - Vertices: 89,438, Triangles: 177,557


In [6]:
# Step 5: Reconstruction Method B - Ball Pivoting Algorithm (BPA)
print("Running Ball-Pivoting Algorithm (BPA)...")
distances = pcd_clean.compute_nearest_neighbor_distance()
avg_dist = np.mean(distances)
radii = [avg_dist * 1.5, avg_dist * 3.0, avg_dist * 6.0]

bpa_mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
    pcd_clean, o3d.utility.DoubleVector(radii)
)

bpa_mesh.remove_degenerate_triangles()
bpa_mesh.remove_duplicated_triangles()
bpa_mesh.remove_duplicated_vertices()
bpa_mesh.remove_non_manifold_edges()

print(f"BPA Mesh - Vertices: {len(bpa_mesh.vertices):,}, Triangles: {len(bpa_mesh.triangles):,}")

Running Ball-Pivoting Algorithm (BPA)...
BPA Mesh - Vertices: 131,582, Triangles: 111,390


In [7]:
# Step 6: Transfer Colors from Point Cloud to Mesh
# Build KDTree to query nearest point colors for Poisson mesh vertices
pcd_tree = o3d.geometry.KDTreeFlann(pcd_clean)
mesh_vertices = np.asarray(poisson_mesh.vertices)
mesh_colors = []

pcd_colors_arr = np.asarray(pcd_clean.colors)
for v in mesh_vertices:
    [_, idx, _] = pcd_tree.search_knn_vector_3d(v, 1)
    mesh_colors.append(pcd_colors_arr[idx[0]])

poisson_mesh.vertex_colors = o3d.utility.Vector3dVector(np.array(mesh_colors))
print("Vertex colors transferred successfully to Poisson mesh!")

Vertex colors transferred successfully to Poisson mesh!


In [8]:
# Step 7: Export Reconstructed Meshes
poisson_out = os.path.join(OUTPUT_DIR, "3dgs_poisson_mesh.ply")
bpa_out = os.path.join(OUTPUT_DIR, "3dgs_bpa_mesh.ply")
poisson_obj = os.path.join(OUTPUT_DIR, "3dgs_poisson_mesh.obj")

o3d.io.write_triangle_mesh(poisson_out, poisson_mesh)
o3d.io.write_triangle_mesh(bpa_out, bpa_mesh)
o3d.io.write_triangle_mesh(poisson_obj, poisson_mesh)

print(f"Saved Poisson Mesh to: {poisson_out}")
print(f"Saved Poisson OBJ to:  {poisson_obj}")
print(f"Saved BPA Mesh to:     {bpa_out}")

Saved Poisson Mesh to: ../outputs/mesh_from_3dgs/3dgs_poisson_mesh.ply
Saved Poisson OBJ to:  ../outputs/mesh_from_3dgs/3dgs_poisson_mesh.obj
Saved BPA Mesh to:     ../outputs/mesh_from_3dgs/3dgs_bpa_mesh.ply


In [9]:
# Step 8: Interactive 3D Mesh Inspection (Plotly)
# Downsample mesh slightly for fast web rendering in notebook
mesh_disp = poisson_mesh.simplify_quadric_decimation(target_number_of_triangles=20000)
verts = np.asarray(mesh_disp.vertices)
tris = np.asarray(mesh_disp.triangles)
v_colors = np.asarray(mesh_disp.vertex_colors)

# Convert RGB colors to hex strings for Plotly
hex_colors = [f"rgb({int(c[0]*255)},{int(c[1]*255)},{int(c[2]*255)})" for c in v_colors]

fig = go.Figure(data=[
    go.Mesh3d(
        x=verts[:, 0],
        y=verts[:, 1],
        z=verts[:, 2],
        i=tris[:, 0],
        j=tris[:, 1],
        k=tris[:, 2],
        vertexcolor=hex_colors,
        lighting=dict(ambient=0.6, diffuse=0.8, roughness=0.5, specular=0.2),
        lightposition=dict(x=100, y=200, z=100)
    )
])

fig.update_layout(
    title="Interactive 3D Mesh from 3DGS (Poisson Reconstruction)",
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode="data"
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    height=650
)
fig.show()